# Dev30 - SVM Classification on 028 Transect (Cropped Regions)

**Goal:** Train SVM on full 028 transect ROIs, then classify only the 4-5 cropped regions of interest.

**Complete pipeline with proper naming:**
1. **Preprocessing** → Crop wavelengths → Smooth → (Normalize commented out for now)
2. **ROI Mapping** → Map ROI names to main classes (sediment, rust, dark_pit, halo)
3. **Training** → Train SVM on full 028 transect with spatial cross-validation
4. **Classification** → Apply SVM only to 4-5 cropped regions
5. **Visualization** → Plot results for each crop with ROI overlays

**Two Classification Modes:**
- **Binary:** sediment vs rust (with confidence threshold → uncertain class)
- **Multi-class:** sediment vs rust vs dark_pit vs halo (with confidence threshold → uncertain class)

**ROI Mapping:**
- `sediment` → sediment
- `rust` → rust
- `1_dark`, `2_dark`, `3_dark`, `dark_pits` → dark_pit
- `1_halo`, `2_halo`, `3_halo` → halo

**Crop Regions (from dev27/dev29):**
- Crop 1: track=1258, slit=212, width=400
- Crop 2: track=5592, slit=765, width=500
- Crop 3: track=5160, slit=613, width=400
- Crop 4: track=717, slit=385, width=400
- (Crop 5: track=4026, slit=582, width=400 - optional)

**Key features:**
- ✅ **Wavelength cropping**: 490-680 nm range
- ✅ **Moving average smoothing**: Window=10 (or Gaussian σ=5)
- ✅ **L2 normalization**: COMMENTED OUT (test without first)
- ✅ **Spatial cross-validation**: 5 folds
- ✅ **Hyperparameter optimization**: Grid search for C and gamma
- ✅ **Confidence threshold**: Reject low-confidence predictions → uncertain class
- ✅ **Crop-only classification**: Classify only regions of interest (not full transect)

**Date:** November 5, 2025

## Setup and Import

In [ ]:
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../"))

# Force complete module reload by removing from cache
if "utils.gref_pipeline.georef" in sys.modules:
    del sys.modules["utils.gref_pipeline.georef"]
if "utils.gref_pipeline" in sys.modules:
    del sys.modules["utils.gref_pipeline"]
if "gref_pipeline.georef" in sys.modules:
    del sys.modules["gref_pipeline.georef"]

from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

print("✅ Module reloaded with cache cleared!")

## Load Data

In [ ]:
# Load 028 transect
transect = load_transect(r"E:\mjosa_new_oct_2025\use_gref4hsi\028\output")
transect.list_files()

# Select all 5 files from transect 028
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

## Import and Map ROIs

Load ROIs from dev27 and map them to main classes.

In [ ]:
# Import ROIs
cube.import_rois("./ROIs/028_new.json")
cube.list_rois()

In [ ]:
# Define ROI mapping to main classes
roi_mapping = {
    # Binary classification ROIs
    "sediment": "sediment",
    "rust": "rust",
    # Multi-class: dark bombs (numbered dark areas around bombs)
    "1_dark": "dark_bomb",
    "2_dark": "dark_bomb",
    "3_dark": "dark_bomb",
    # Multi-class: dark pits (separate dark pit class)
    "dark_pits": "dark_pit",
    "dark_pits_multiple": "dark_pit",  # Additional dark pit ROI (same class as dark_pits)
    # Multi-class: halos
    "1_halo": "halo",
    "2_halo": "halo",
    "3_halo": "halo",
}

# Define training ROIs for each classification mode
training_rois_binary = ["sediment", "rust"]

training_rois_multiclass = [
    "sediment",
    "rust",
    "1_dark",
    "2_dark",
    "3_dark",
    "dark_pits",
    "dark_pits_multiple",  # Additional dark pit ROI (same class as dark_pits)
    "1_halo",
    "2_halo",
    "3_halo",
]

print("📋 ROI Mapping:")
for roi, mapped in roi_mapping.items():
    if roi in cube.roi_collection:
        print(f"   {roi} → {mapped} ({len(cube.roi_collection[roi])} pixels)")

print(f"\n🎯 Binary classification ROIs: {training_rois_binary}")
print(f"🎯 Multi-class classification ROIs: {training_rois_multiclass}")

## Visualize ROIs on Full Transect

In [ ]:
%matplotlib inline

# Plot all training ROIs for multi-class
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(5, 50),
    show_file_boundaries=False,
    roi_collection=training_rois_multiclass,
    roi_legend_loc="outside",
    roi_marker_size=2,
    roi_legend_markersize=50,
    roi_marker_edgewidth=0,
)

## Define Crop Regions

Define the 4-5 cropped regions where classification will be applied.

In [ ]:
# Define crop regions (UPDATED coordinates from user)
crop_regions = [
    {
        "name": "Crop 1 (Bomb 2 - Rust)",
        "track": 1258,
        "slit": 250,  # UPDATED from 212
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 2 (Bomb 3)",
        "track": 5592,
        "slit": 700,  # UPDATED from 765
        "width": 500,
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 3 (Bomb 1)",
        "track": 5160,
        "slit": 613,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 4 (Dark pits)",
        "track": 717,
        "slit": 385,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
]

print(f"📍 Defined {len(crop_regions)} crop regions (UPDATED):")
for i, crop in enumerate(crop_regions, 1):
    print(
        f"   {i}. {crop['name']}: track={crop['track']}, slit={crop['slit']}, width={crop['width']}"
    )

## Visualize Crop Regions with ROIs

### 📋 Plot Specifications for Crop Regions

**All cropped region plots must use these settings:**
- `figsize=(30, 8.32)` - Wide horizontal layout
- `crop_width=500` - All crops same width
- `crop_aspect_ratio=4` - Height = width/4
- `display_aspect_ratio=4.0` - Display stretching
- `roi_marker_size=2` - Small markers for ROIs
- `roi_legend_markersize=50` - Larger legend markers
- `roi_marker_edgewidth=0` - No marker edges
- `flip_axes=True`, `flip_horizontal=True` - Standard orientation
- `show_file_boundaries=False` - Clean view

### ✅ Plot Verification Checklist

**All `plot_rgb` calls have been verified and corrected:**

1. ✅ **Cell: "Visualize ROIs on Full Transect"** - Full transect view (roi_marker_size=2) ✓
2. ✅ **Cell: "TEST crop boundary"** - FIXED: figsize=(30, 8.32), aspect=4, display_aspect=4.0 ✓
3. ✅ **Cell: "Visualize each crop region loop"** - FIXED: roi_marker_size=2, all specs correct ✓
4. ✅ **Cell: "Sediment distribution diagnostic"** - Full transect view (roi_marker_size=2) ✓
5. ✅ **Cell: "Filtered training ROIs"** - Full transect view (roi_marker_size=2) ✓
6. ✅ **Function: `plot_crop_classification()`** - FIXED: All crop specs (30x8.32, roi_marker_size=2, display_aspect=4.0) ✓

**Summary:** All 6 `plot_rgb` instances now follow the correct specifications!

In [ ]:
# TEST: Plot crop that extends beyond image boundary
# Should show GLOBAL coordinates in extent (can be negative)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(30, 8.32),  # FIXED - was (8, 8)
    crop_center_track=1258,
    crop_center_slit=150,  # Near edge - crop will extend beyond boundary
    crop_width=600,
    crop_aspect_ratio=4,  # FIXED - was 3.5
    display_aspect_ratio=4.0,  # ADDED
    show_file_boundaries=False,
)
print(
    "\n✅ Expected: Slit extent should be [-150, 450] (centered at 150 with width 600)"
)
print("   Even though image only has slits [0, 968], extent shows requested range")

In [ ]:
# Visualize each crop region with relevant ROIs (UPDATED settings)
for crop in crop_regions:
    print(f"\n📷 {crop['name']}")
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),  # UPDATED
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,  # UPDATED - Controls display stretching
        show_file_boundaries=False,
        roi_collection=training_rois_multiclass,
        roi_legend_loc="outside",
        roi_marker_size=2,  # FIXED - was 200
        roi_legend_markersize=50,
        roi_marker_edgewidth=0,
    )

## Preprocessing Pipeline

Apply same preprocessing as dev22.

### Step 1: Crop Wavelengths (490-680 nm)

In [ ]:
# Apply smoothing BEFORE wavelength filter (force recompute to avoid cache issues)
cube.apply_spectral_smoothing(
    method="gaussian",
    gaussian_sigma=5.0,
)

In [ ]:
# Crop wavelengths to 490-700 nm (UPDATED)
cube.apply_wavelength_filter(wavelength_range=(490, 700))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")

### Step 2: Spectral Smoothing (Moving Average, Window=10)

## ⚠️ IMPORTANT: Rerun preprocessing after module reload!

When you reload the module, the `cube` object persists but its methods are outdated. You need to **re-apply all preprocessing** to use the new module code:
1. Illumination correction
2. Wavelength filter  
3. Spectral smoothing

### Step 3: L2 Normalization (COMMENTED OUT FOR NOW)

In [ ]:
# # Apply L2 normalization (unit-length spectra)
# cube.apply_spectral_normalization(method="l2")

# print(f"✅ L2 normalization complete")
# print(
#     f"   Value range: [{cube.data_corrected.min():.4f}, {cube.data_corrected.max():.4f}]"
# )

### Visualize Preprocessed Spectra

In [ ]:
# Plot preprocessed spectra for multi-class training ROIs
cube.plot_spectrum(
    roi_names=training_rois_multiclass,
    use_corrected=True,
    wavelength_range=(
        490,
        700,
    ),  # Filter to 490-680 nm (don't rely on apply_wavelength_filter)
    # wavelength_smoothing=1,  # No additional smoothing
    normalize_method=None,  # No additional normalization
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
)

## 🐛 **DATA CORRUPTION ISSUE**

The `cube` data is now corrupted because:
1. You ran `apply_wavelength_filter()` which modified `self.wavelengths` to 108 elements
2. You reloaded the module, which reset the methods but NOT the data
3. Now `self.wavelengths` has 108 elements but `self.data_corrected` has 210 wavelengths

**FIX:** Run cells 4-6 again to reload the transect with fresh data (starting from "Load 028 transect")

In [ ]:
# DEBUG: Check data shapes after wavelength filtering
print("🔍 DEBUG: Data shapes after apply_wavelength_filter()")
print(f"   self.wavelengths: {len(cube.wavelengths)} elements")
print(f"   self.data shape: {cube.data.shape}")
print(f"   self.data_corrected shape: {cube.data_corrected.shape}")
print(f"\n   Expected: All should have 108 wavelengths (490-680 nm)")
print(f"   Issue: data_corrected might have 108 but data might still have 210!")

---

# PART 1: Binary Classification (Sediment vs Rust)

---

## Train SVM (Binary: Sediment vs Rust)

## 🐛 Diagnose Spatial Grouping Issue

The NaN values in CV results indicate that spatial grouping created too few groups (sediment has 0 groups). Let's investigate why and fix the parameters.

---

## 📋 Summary: Understanding the NaN CV Results Issue

**Problem:** The cross-validation results show `NaN` values for all metrics.

**Root Cause:** 
- Sediment class created **0 spatial groups** (all pixels marked as "sediment_dropped")
- Rust class created **1 spatial group** (58 pixels)
- With only 1-2 groups total, cross-validation cannot split into training/validation folds
- Result: All folds skipped → empty metrics → NaN

**Why did sediment create 0 groups?**
- Grid-based grouping (`use_grid_for_sediment=True`) is enabled by default
- Sediment pixels are likely very scattered across the transect
- Grid tiles are too large (200×100 pixels) → most tiles have < 10 pixels (min threshold)
- All tiles get merged into "sediment_dropped"

**Solution:**
- **Option 1:** Disable grid-based grouping (`use_grid_for_sediment=False`) → Use connected components
- **Option 2:** Reduce grid size (e.g., 50×50) → More pixels per tile
- **Option 3:** Lower `min_group_size` threshold (e.g., 5 instead of 10)

Run the diagnostic cell below to analyze sediment distribution.

---

In [ ]:
# Diagnose: Check sediment ROI spatial distribution
import numpy as np

sediment_pixels = cube.roi_collection["sediment"]
print(f"📊 Sediment ROI Analysis:")
print(f"   Total pixels: {len(sediment_pixels)}")

# Extract track and slit coordinates
slits = [p[0] for p in sediment_pixels]
tracks = [p[1] for p in sediment_pixels]

print(f"\n📏 Spatial extent:")
print(
    f"   Track range: [{min(tracks)}, {max(tracks)}] (span: {max(tracks) - min(tracks)})"
)
print(f"   Slit range: [{min(slits)}, {max(slits)}] (span: {max(slits) - min(slits)})")

# Calculate density
track_span = max(tracks) - min(tracks) + 1
slit_span = max(slits) - min(slits) + 1
total_area = track_span * slit_span
density = len(sediment_pixels) / total_area if total_area > 0 else 0

print(f"\n🎯 Density:")
print(f"   Bounding box area: {total_area} pixels ({track_span} × {slit_span})")
print(f"   Density: {density:.4f} ({len(sediment_pixels)}/{total_area})")

# Grid analysis
grid_track = 200  # Current setting
grid_slit = 100  # Current setting
n_grid_cells_track = track_span // grid_track + 1
n_grid_cells_slit = slit_span // grid_slit + 1
expected_tiles = n_grid_cells_track * n_grid_cells_slit

print(f"\n📐 Grid analysis (current settings):")
print(f"   Grid size: {grid_track}(track) × {grid_slit}(slit)")
print(
    f"   Expected tiles: ~{expected_tiles} ({n_grid_cells_track} × {n_grid_cells_slit})"
)
print(f"   Avg pixels per tile: {len(sediment_pixels) / expected_tiles:.1f}")

print(f"\n💡 Recommendation:")
if density < 0.1:
    print(f"   ⚠️ Low density ({density:.4f}) - sediment is very scattered!")
    print(f"   Consider: DISABLING grid-based grouping (use_grid_for_sediment=False)")
    print(f"   Or: Reduce grid size to get more pixels per tile")
else:
    print(f"   ✅ Density looks reasonable")

# Visualize sediment distribution
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(5, 50),
    roi_collection=["sediment"],
    roi_marker_size=2,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## 🔧 Fixed Training: Disable Grid-Based Grouping

Based on diagnosis, let's try disabling grid-based grouping for sediment and use connected components instead.

In [ ]:
## FIX: Train SVM with spatial cross-validation - INCREASED CLOSING RADIUS
# Sediment: connected components (closing_radius=50) - merges scattered pixels
# Rust: Will remain as 1 group (compact 58-pixel blob)
cv_results_binary_fixed = cube.train_svm_with_cv(
    training_rois=training_rois_binary,
    segment_start=None,  # None = use all data
    segment_end=None,  # None = use all data
    wavelength_range=None,  # Already preprocessed
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    add_brightness_feature=False,
    use_intensity_only=False,
    # 🔥 FIX: Increased closing radius to merge scattered sediment pixels
    use_grid_for_sediment=False,  # Sediment: connected components (scattered)
    closing_radius=50,  # INCREASED from 5 to 50 - merges scattered pixels
    min_group_size=20,  # Minimum pixels per group
    quiet=False,
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 BINARY CLASSIFICATION CV SUMMARY (FIXED)")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results_binary_fixed['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results_binary_fixed['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results_binary_fixed['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results_binary_fixed['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results_binary_fixed['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results_binary_fixed['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results_binary_fixed['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results_binary_fixed['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results_binary_fixed['best_params']['C']}")
print(f"  gamma = {cv_results_binary_fixed['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results_binary_fixed["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

In [ ]:
# Plot spatial groups created with fixed settings
if (
    "spatial_groups_roi_collection" in cv_results_binary_fixed
    and cv_results_binary_fixed["spatial_groups_roi_collection"]
):
    print("\n📊 Spatial Groups Created (Fixed Settings):")
    for group_name, pixels in sorted(
        cv_results_binary_fixed["spatial_groups_roi_collection"].items()
    ):
        print(f"   {group_name}: {len(pixels)} pixels")

    cube.plot_georef(
        use_corrected=True,
        figsize=(40, 10),
        roi_collection=cv_results_binary_fixed["spatial_groups_roi_collection"],
        roi_marker_size=3,
        roi_legend_loc="outside",
        roi_marker_edgewidth=0,
        roi_legend_markersize=40,
    )
else:
    print("⚠️ No spatial groups found in cv_results_binary_fixed.")

In [ ]:
# Train SVM with spatial cross-validation - BINARY MODE
# BUG FIX: Use None for segment_start/end to include ALL data (not exclude)
cv_results_binary = cube.train_svm_with_cv(
    training_rois=training_rois_binary,
    segment_start=None,  # None = use all data
    segment_end=None,  # None = use all data
    wavelength_range=None,  # Already preprocessed
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    add_brightness_feature=False,
    use_intensity_only=False,
    quiet=False,
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 BINARY CLASSIFICATION CROSS-VALIDATION SUMMARY")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results_binary['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results_binary['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results_binary['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results_binary['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results_binary['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results_binary['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results_binary['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results_binary['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results_binary['best_params']['C']}")
print(f"  gamma = {cv_results_binary['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results_binary["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

## Visualize Spatial Groups Created for CV

Visualize the spatial groups that were created for cross-validation. Each group represents pixels that should stay together during CV splits.

In [ ]:
# Plot spatial groups created during training
if (
    "spatial_groups_roi_collection" in cv_results_binary
    and cv_results_binary["spatial_groups_roi_collection"]
):
    print("\n📊 Spatial Groups Created for Binary Classification:")
    for group_name, pixels in sorted(
        cv_results_binary["spatial_groups_roi_collection"].items()
    ):
        print(f"   {group_name}: {len(pixels)} pixels")

    cube.plot_georef(
        use_corrected=True,
        figsize=(40, 10),
        roi_collection=cv_results_binary["spatial_groups_roi_collection"],
        roi_marker_size=3,
        roi_legend_loc="outside",
        roi_marker_edgewidth=0,
        roi_legend_markersize=40,
    )
else:
    print(
        "⚠️ No spatial groups found in cv_results_binary. Training may not have used spatial clustering."
    )

In [ ]:
# Visualize filtered training ROIs
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(5, 50),
    roi_collection=cv_results_binary["filtered_training_rois"],
    roi_marker_size=2,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
)

## Classify Crop Regions (Binary)

Apply classification only to the cropped regions of interest.

In [ ]:
# Helper function to classify a single crop region
def classify_crop_region(cube, crop, confidence_threshold=0.5):
    """Classify a cropped region and return results."""
    # Calculate crop bounds
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    crop_track_min = max(0, crop["track"] - half_width_track)
    crop_track_max = min(cube.data_corrected.shape[0], crop["track"] + half_width_track)
    crop_slit_min = max(0, crop["slit"] - half_width_slit)
    crop_slit_max = min(cube.data_corrected.shape[1], crop["slit"] + half_width_slit)

    print(f"\n🔍 Classifying: {crop['name']}")
    print(
        f"   Crop bounds: track [{crop_track_min}:{crop_track_max}], slit [{crop_slit_min}:{crop_slit_max}]"
    )

    # Classify the segment
    classification_results = cube.classify_segment(
        segment_start=crop_track_min,
        segment_end=crop_track_max - 1,
        use_corrected=True,
        confidence_threshold=confidence_threshold,
        quiet=False,
    )

    # Print pixel counts
    print(f"\n📊 Pixel counts for {crop['name']}:")
    for class_name in classification_results["class_names"]:
        count = np.sum(classification_results["classification_map"] == class_name)
        percentage = 100.0 * count / classification_results["classification_map"].size
        print(f"   {class_name}: {count} pixels ({percentage:.1f}%)")

    return classification_results, crop_track_min, crop_track_max

In [ ]:
# Classify all crop regions (binary mode)
binary_crop_results = []

for crop in crop_regions:
    results, track_min, track_max = classify_crop_region(
        cube, crop, confidence_threshold=0.5
    )
    binary_crop_results.append(
        {
            "crop": crop,
            "results": results,
            "track_min": track_min,
            "track_max": track_max,
        }
    )

### Plot Binary Classification with plot_rgb (includes uncertain class)

Convert classification results to ROI format and visualize with plot_rgb showing all classes including uncertain.

### 🔧 Understanding ROI Marker Size

**How marker size works in both `plot_georef` and `plot_rgb`:**

Both functions use the formula: `s = roi_marker_size**2 * 50`

- `roi_marker_size=1` → `s = 1² × 50 = 50` (tiny dots, not solid pixels)
- `roi_marker_size=2` → `s = 2² × 50 = 200` (solid pixels, like dev20)
- `roi_marker_size=3` → `s = 3² × 50 = 450` (larger overlapping pixels)

**For classified pixels to appear solid (like in dev20), use `roi_marker_size=2` or higher.**

In [ ]:
# Convert binary classification results to ROI format and plot with plot_rgb
print(
    "📊 Plotting binary classification results with plot_rgb (includes uncertain class)..."
)

# For each crop, convert classification_map to ROI collection
for crop_result in binary_crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]
    track_min = crop_result["track_min"]

    # Create ROI collection from classification map
    classification_rois = {}
    for class_name in results["class_names"]:
        mask = results["classification_map"] == class_name
        rows, cols = np.where(mask)
        # Convert to global coordinates (add track_min offset)
        pixels = [(col, row + track_min) for row, col in zip(rows, cols)]
        if len(pixels) > 0:
            classification_rois[class_name] = pixels

    print(f"\n📍 {crop['name']}")
    print(f"   ROIs created: {list(classification_rois.keys())}")
    print(f"   Pixel counts:")
    for class_name in sorted(classification_rois.keys()):
        print(f"      {class_name}: {len(classification_rois[class_name])} pixels")

    # Plot with plot_rgb showing the classified region
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,
        show_file_boundaries=False,
        roi_collection=classification_rois,
        roi_legend_loc="outside",
        roi_marker_size=2,  # FIXED - was 1, now 2 for solid pixels (s=2^2*50=200)
        roi_legend_markersize=50,
        roi_marker_edgewidth=0,
    )

---

# PART 2: Multi-Class Classification (Sediment vs Rust vs Dark vs Halo)

---

## Create Mapped Training ROIs

Map ROIs to main classes for multi-class training.

In [ ]:
# Create new ROI collection with mapped class names
def create_mapped_roi_collection(cube, roi_list, roi_mapping):
    """Create ROI collection with mapped class names."""
    mapped_rois = {}

    for roi_name in roi_list:
        if roi_name in cube.roi_collection:
            mapped_name = roi_mapping.get(roi_name, roi_name)

            # Add pixels to mapped class (merge if class already exists)
            if mapped_name not in mapped_rois:
                mapped_rois[mapped_name] = []
            mapped_rois[mapped_name].extend(cube.roi_collection[roi_name])

    # Remove duplicates
    for class_name in mapped_rois:
        mapped_rois[class_name] = list(set(mapped_rois[class_name]))

    return mapped_rois


# Create mapped ROI collection
mapped_training_rois = create_mapped_roi_collection(
    cube, training_rois_multiclass, roi_mapping
)

print("📋 Mapped training ROIs for multi-class:")
for class_name, pixels in mapped_training_rois.items():
    print(f"   {class_name}: {len(pixels)} pixels")

# Store mapped ROIs temporarily for training
cube._original_roi_collection = cube.roi_collection.copy()  # Backup original ROIs
cube.roi_collection = mapped_training_rois  # Replace with mapped ROIs

## Train SVM (Multi-Class: Sediment vs Rust vs Dark vs Halo)

In [ ]:
# Train SVM with spatial cross-validation - MULTI-CLASS MODE
# BUG FIX: Use None for segment_start/end to include ALL data (not exclude)
cv_results_multiclass = cube.train_svm_with_cv(
    training_rois=list(mapped_training_rois.keys()),  # Use mapped class names
    segment_start=None,  # None = use all data
    segment_end=None,  # None = use all data
    wavelength_range=None,  # Already preprocessed
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    add_brightness_feature=False,
    use_intensity_only=False,
    quiet=False,
    closing_radius=50,  # INCREASED from 5 to 50 - forces connections between scattered pixels
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 MULTI-CLASS CLASSIFICATION CROSS-VALIDATION SUMMARY")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results_multiclass['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results_multiclass['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results_multiclass['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results_multiclass['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results_multiclass['best_params']['C']}")
print(f"  gamma = {cv_results_multiclass['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results_multiclass["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

In [ ]:
# Plot spatial groups for multi-class
if (
    "spatial_groups_roi_collection" in cv_results_multiclass
    and cv_results_multiclass["spatial_groups_roi_collection"]
):
    print("\n📊 Spatial Groups Created for Multi-Class Classification:")
    for group_name, pixels in sorted(
        cv_results_multiclass["spatial_groups_roi_collection"].items()
    ):
        print(f"   {group_name}: {len(pixels)} pixels")

    cube.plot_georef(
        use_corrected=True,
        figsize=(40, 10),
        roi_collection=cv_results_multiclass["spatial_groups_roi_collection"],
        roi_marker_size=3,
        roi_legend_loc="outside",
        roi_marker_edgewidth=0,
        roi_legend_markersize=40,
    )
else:
    print("⚠️ No spatial groups found in cv_results_multiclass.")

In [ ]:
# Restore original ROI collection
cube.roi_collection = cube._original_roi_collection
del cube._original_roi_collection

print("✅ Original ROI collection restored")

## Classify Crop Regions (Multi-Class)

In [ ]:
# Classify all crop regions (multi-class mode)
multiclass_crop_results = []

for crop in crop_regions:
    results, track_min, track_max = classify_crop_region(
        cube, crop, confidence_threshold=0.5
    )
    multiclass_crop_results.append(
        {
            "crop": crop,
            "results": results,
            "track_min": track_min,
            "track_max": track_max,
        }
    )

In [ ]:
# Convert multi-class classification results to ROI format and plot with plot_rgb
print(
    "📊 Plotting multi-class classification results with plot_rgb (includes uncertain class)..."
)

# For each crop, convert classification_map to ROI collection
for crop_result in multiclass_crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]
    track_min = crop_result["track_min"]

    # Create ROI collection from classification map
    classification_rois = {}
    for class_name in results["class_names"]:
        mask = results["classification_map"] == class_name
        rows, cols = np.where(mask)
        # Convert to global coordinates (add track_min offset)
        pixels = [(col, row + track_min) for row, col in zip(rows, cols)]
        if len(pixels) > 0:
            classification_rois[class_name] = pixels

    print(f"\n📍 {crop['name']}")
    print(f"   ROIs created: {list(classification_rois.keys())}")
    print(f"   Pixel counts:")
    for class_name in sorted(classification_rois.keys()):
        print(f"      {class_name}: {len(classification_rois[class_name])} pixels")

    # Plot with plot_rgb showing the classified region
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,
        show_file_boundaries=False,
        roi_collection=classification_rois,
        roi_legend_loc="outside",
        roi_marker_size=2,  # FIXED - was 1, now 2 for solid pixels (s=2^2*50=200)
        roi_legend_markersize=50,  # FIXED - was 1
        roi_marker_edgewidth=0,
    )

### 🎨 Proper Classification Visualization

**Previous approach was WRONG:** Using ROI markers creates artificial symbols stacked on top of RGB image.

**Correct approach:** Create a true classified image where each pixel has the color of its class - like any normal image!

In [ ]:
def plot_classification_as_image(
    cube,
    crop,
    classification_map,
    class_names,
    figsize=(30, 8.32),
    flip_axes=True,
    flip_horizontal=True,
):
    """
    Plot classification results as a proper classified image (not ROI markers).
    Each pixel gets the color of its class - like a normal image.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    # Define colors for each class
    class_colors = {
        "sediment": [0.6, 0.4, 0.2],  # Brown
        "rust": [0.8, 0.2, 0.1],  # Red-orange
        "dark_bomb": [0.1, 0.1, 0.1],  # Dark gray/black
        "dark_pit": [0.2, 0.2, 0.2],  # Gray
        "halo": [0.9, 0.9, 0.5],  # Yellow
        "uncertain": [0.5, 0.5, 0.5],  # Medium gray
    }

    # Create RGB image from classification map
    height, width = classification_map.shape
    rgb_image = np.zeros((height, width, 3))

    for class_name in class_names:
        mask = classification_map == class_name
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])  # Default gray
        rgb_image[mask] = color

    # Apply flipping to match plot_rgb behavior
    if flip_axes:
        rgb_image = rgb_image.transpose(1, 0, 2)  # Swap track and slit
    if flip_horizontal:
        rgb_image = rgb_image[:, ::-1, :]  # Flip horizontally

    # Calculate extent in global coordinates
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    slit_min = crop["slit"] - half_width_slit
    slit_max = crop["slit"] + half_width_slit
    track_min = crop["track"] - half_width_track
    track_max = crop["track"] + half_width_track

    if flip_axes:
        # When axes flipped: x=track, y=slit
        extent = [track_min, track_max, slit_min, slit_max]
        xlabel, ylabel = "Track Index", "Slit Index"
    else:
        # Normal: x=slit, y=track
        extent = [slit_min, slit_max, track_min, track_max]
        xlabel, ylabel = "Slit Index", "Track Index"

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Display the classified image
    im = ax.imshow(
        rgb_image,
        extent=extent,
        origin="lower",
        aspect=crop.get("display_aspect_ratio", 4.0),
        interpolation="nearest",
    )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"Classification: {crop['name']}", fontsize=14, fontweight="bold")

    # Create legend with class colors
    legend_patches = []
    for class_name in class_names:
        count = np.sum(classification_map == class_name)
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])
        label = f"{class_name} ({count} px)"
        legend_patches.append(mpatches.Patch(color=color, label=label))

    ax.legend(
        handles=legend_patches,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=True,
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    return fig, ax


# Plot multi-class classification results as proper images
print("📊 Plotting multi-class classification as IMAGES (not ROI markers)...")

for crop_result in multiclass_crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]

    print(f"\n📍 {crop['name']}")
    print(f"   Classes: {results['class_names']}")
    print(f"   Pixel counts:")
    for class_name in results["class_names"]:
        count = np.sum(results["classification_map"] == class_name)
        percentage = 100.0 * count / results["classification_map"].size
        print(f"      {class_name}: {count} pixels ({percentage:.1f}%)")

    # Plot as proper classified image
    plot_classification_as_image(
        cube,
        crop,
        results["classification_map"],
        results["class_names"],
        figsize=(30, 8.32),
        flip_axes=True,
        flip_horizontal=True,
    )

---

## Summary and Next Steps

**Completed:**
- ✅ Loaded 028 transect and applied illumination correction
- ✅ Imported and mapped ROIs to main classes
- ✅ Applied preprocessing (wavelength crop, smoothing)
- ✅ Trained binary SVM (sediment vs rust) with spatial CV
- ✅ Trained multi-class SVM (sediment vs rust vs dark_pit vs halo) with spatial CV
- ✅ Classified 4-5 cropped regions (not full transect)
- ✅ Visualized classification results for each crop

**Key Findings:**
- Binary classification performance: [See CV results above]
- Multi-class classification performance: [See CV results above]
- Confidence threshold creates "uncertain" class for ambiguous pixels

**Next Steps:**
1. Adjust crop boundaries (track/slit/width) if needed
2. Test different confidence thresholds (0.3, 0.5, 0.7)
3. Enable L2 normalization and compare results
4. Apply filtering (morphological operations) if needed
5. Validate against ground truth ROIs

**Notes:**
- Spatial cross-validation ensures test pixels are spatially separated from training
- Hyperparameters (C, gamma) are automatically optimized via grid search
- Classification is applied ONLY to crop regions (efficient!)
- Full transect is used for training (more diverse examples)